# T61 — Debugging a plate model: topology construction anomalies

**Cluster M: Plate-model debugging** (3 / 4).

## What this notebook does

T59 and T60 test whether resolved plate boundaries move the way their topology declares (right sign of velocity, plausible magnitudes). T61 goes one level deeper and tests the CONSTRUCTION of the topology itself — the graph of how boundary sections are stitched into plate polygons. Three distinct anomaly types are detected:

1. **Gaps and overlaps** — every point on Earth's surface should belong to EXACTLY ONE resolved topology. Points that belong to zero (gaps) or two-or-more (overlaps) are construction errors. This detector finds shared boundary sub-segments that are shared by a number of topologies other than 2.

2. **Non-unique sections** — a topological polygon boundary is a graph of section references. Ideally, each unique boundary section is referenced once per polygon. There ARE legitimate cases of double-referencing (e.g. head-of-A + B + tail-of-A + C forms a valid polygon), but when the two references correspond to sub-segments that TOUCH each other, that's a duplication error.

3. **Missing subduction polarity** — every subduction-zone feature used in a topology needs a `gpml_subduction_polarity` property specifying which side of the trench is subducting. When it's missing (or set to "Unknown"), downstream tools can't figure out which way the slab dips, and feature-extraction workflows silently drop those trenches.

All three detectors return per-time `pygplates.Feature` lists (locating each anomaly) that this notebook renders as pyGMT overlays on the plate topology.

**Sections:**
1. **§1** Load plate model + refactored `detect_anomalies` module.
2. **§2** For each snapshot (100, 125, 150, 175, 200 Ma) run all three detectors; report counts per age.
3. **§3** Render one pyGMT map per snapshot showing all three anomaly types on the same base map.
4. **§4** Compact MP4 video at 1-Myr cadence.
5. **§5** Extend this — deeper time windows, filter by anomaly type, GPML export for GPlates.

**Reading the results.** No plate model is fully clean of these construction anomalies — the point is not to have zero flags, but to know WHERE they are so you can decide whether to fix them or work around them for your particular analysis. A subduction-zone anomaly in a region you're not studying is fine to leave; a gap that intersects your sample locations is not.

**Audience**: intermediate → researcher.
**Difficulty**: ★★★.
**Runtime**: ~1 min for the 5 snapshots; ~5-8 min for the 1-Myr video.


## Data availability

**Plate model** — Zahirovic 2022, fetched via PMM. Any PMM model can be swapped.

**Helper module** — `Notebooks/plate_model_debug/detect_anomalies.py` (refactored from Ben's two standalone scripts into per-time functions).

## Sources

- Sculley, B. & Cannon, J. (2025) *plate-model-debug*. EarthByte, University of Sydney.
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. https://doi.org/10.1002/gdj3.146


## Environment + imports


In [ ]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import gplately
import pygmt
import pygplates
from plate_model_manager import PlateModelManager

sys.path.insert(0, str(Path("Notebooks").resolve()))
from plate_model_debug import (
    detect_gaps_and_overlaps_at_time,
    detect_non_unique_sections_at_time,
    detect_missing_polarity_at_time,
    calculate_plate_motion_arrows,
)

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, gplately, pygmt, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


In [ ]:
# === USER CONFIGURATION =====================================================
MODEL_NAME             = "Zahirovic2022"
ANCHOR_PLATE_ID        = 0

SNAPSHOT_AGES_MA       = [100, 125, 150, 175, 200]

VIDEO_START_MA         = 200
VIDEO_END_MA           = 100
VIDEO_CADENCE_MA       = 1
VIDEO_FPS              = 10
VIDEO_WIDTH_PX         = 800
VIDEO_CRF              = 28

# The non-unique-sections detector treats two sub-segments as "touching"
# if they are within this threshold (km). Ben's default: 100 m.
NON_UNIQUE_TOUCH_KM    = 0.1

# Whether "Unknown" polarity counts as anomalous (missing is always anomalous).
POLARITY_UNKNOWN_IS_ANOMALOUS = True

REGION_GLOBAL          = [-180, 180, -75, 80]
PROJECTION             = "N15c"

FRAMES_DIR             = Path("Notebooks/T61_debug_topology_frames")
VIDEO_DIR              = Path("Notebooks/videos")
VIDEO_PATH             = VIDEO_DIR / "T61_debug_topology.mp4"
# Plate-motion vector overlay.
ARROW_SPACING_DEG      = 15.0     # grid spacing for arrow sampling (global 15° = readable, regional 3-5°)
ARROW_VEL_SCALE        = 0.08     # cm map length per cm/yr (map-projection-dependent; tune to taste)
ARROW_MIN_SPEED_CM_YR  = 0.5      # arrows below this speed are suppressed (removes near-stationary noise)

# ============================================================================
print(f"  plate model:  {MODEL_NAME}")
print(f"  snapshots:    {SNAPSHOT_AGES_MA} Ma")
print(f"  video window: {VIDEO_END_MA}-{VIDEO_START_MA} Ma at {VIDEO_CADENCE_MA}-Myr cadence")


## 1. Load plate model + wrap into a TopologicalModel


In [ ]:
pmm = PlateModelManager()
model_pmm = pmm.get_model(MODEL_NAME, data_dir="./gplately_data")

_pmm_rot = model_pmm.get_rotation_model()
if not hasattr(_pmm_rot, "get_rotation"):
    rotation_model = pygplates.RotationModel(_pmm_rot)
else:
    rotation_model = _pmm_rot

topology_features = [pygplates.FeatureCollection(f) for f in model_pmm.get_topologies()]

# TopologicalModel wrapper is required for the non-unique-sections detector
# (introduced in pyGPlates 0.43+).
topological_model = pygplates.TopologicalModel(topology_features, rotation_model)

recon = gplately.PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=model_pmm.get_topologies(),
    static_polygons=model_pmm.get_static_polygons(),
    anchor_plate_id=ANCHOR_PLATE_ID,
)
print(f"  plate model loaded: {MODEL_NAME}")
print(f"  topology feature collections: {len(topology_features)}")


## 2. Run all three detectors at each snapshot

Print per-anomaly counts per age so you can see the raw scale of the problem before drawing anything.


In [ ]:
def run_all_detectors(time):
    """Return (gaps_overlaps, non_unique_topos, non_unique_sections, missing_polarity)."""
    gaps = detect_gaps_and_overlaps_at_time(rotation_model, topology_features, time)
    nu_topos, nu_sections = detect_non_unique_sections_at_time(
        topological_model, time, touch_threshold_km=NON_UNIQUE_TOUCH_KM)
    missing_pol = detect_missing_polarity_at_time(
        rotation_model, topology_features, time,
        unknown_is_anomalous=POLARITY_UNKNOWN_IS_ANOMALOUS)
    return gaps, nu_topos, nu_sections, missing_pol

print(f"{'age (Ma)':>10}  {'gaps/overlaps':>14}  {'non-unique topos':>17}  "
      f"{'non-unique subs':>16}  {'missing polarity':>17}")
print("-" * 78)
for _age in SNAPSHOT_AGES_MA:
    gaps, nu_t, nu_s, mp = run_all_detectors(float(_age))
    print(f"  {_age:>7} Ma  {len(gaps):>14}  {len(nu_t):>17}  {len(nu_s):>16}  {len(mp):>17}")


## 3. Static snapshots — all three anomaly types on one map

Colour coding:

- **Red thick lines** — sub-segments locating gaps and overlaps
- **Blue thick lines** — sub-segments that are non-unique (same section added twice with touching duplicates)
- **Orange dots** — subduction-zone features with missing / unknown polarity (plotted as feature centroids)


In [ ]:
def render_anomaly_map(time, out_path=None, width_cm=15):
    gaps, nu_topos, nu_sections, missing_pol = run_all_detectors(time)

    gplot = gplately.PlotTopologies(
        plate_reconstruction=recon,
        coastlines=model_pmm.get_coastlines(),
        continents=model_pmm.get_continental_polygons(),
        COBs=model_pmm.get_COBs(),
        time=float(time),
        plot_engine=gplately.PygmtPlotEngine(),
    )

    fig = pygmt.Figure()
    fig.basemap(region=REGION_GLOBAL, projection=f"N{width_cm}c", frame=["af", "WSne"])

    try:
        gplot.plot_continents(fig, fill="gray95", pen="0.2p,gray40")
        gplot.plot_coastlines(fig, pen="0.3p,gray20")
    except Exception:
        pass

    try:
        engine = gplately.PygmtPlotEngine()
        engine.plot_geo_data_frame(fig, gplot.get_all_topological_sections(),
                                    pen="0.4p,gray60")
    except Exception:
        pass

    # --- Plate-motion vector overlay (dark-grey arrows under the anomaly layer)
    try:
        _snap = topological_model.topological_snapshot(float(time))
        _alons, _alats, _ae, _an = calculate_plate_motion_arrows(
            _snap, region=REGION_GLOBAL, spacing_deg=ARROW_SPACING_DEG)
        if len(_alons):
            _speed = np.sqrt(_ae ** 2 + _an ** 2)
            _azimuth = np.rad2deg(np.arctan2(_ae, _an))
            _m = _speed >= ARROW_MIN_SPEED_CM_YR
            if _m.any():
                fig.plot(x=_alons[_m], y=_alats[_m],
                          style="V0.15c+e+a35",
                          direction=[_azimuth[_m], ARROW_VEL_SCALE * _speed[_m]],
                          fill="gray25", pen="0.3p,gray25")
    except Exception as _e:
        print(f"    (skip velocity arrows: {type(_e).__name__}: {_e})")


    # Anomaly line features (gaps + non-unique)
    def _plot_line_features(features, pen):
        for f in features:
            for geom in f.get_geometries():
                pts = geom.to_lat_lon_array()
                if len(pts) > 1:
                    fig.plot(x=pts[:, 1], y=pts[:, 0], pen=pen)

    _plot_line_features(gaps, pen="2p,red")
    _plot_line_features(nu_sections, pen="2p,blue")

    # Missing-polarity features rendered as centroid dots
    if missing_pol:
        _lons, _lats = [], []
        for f in missing_pol:
            for geom in f.get_geometries():
                pts = geom.to_lat_lon_array()
                if len(pts) > 0:
                    _lons.append(float(pts[:, 1].mean()))
                    _lats.append(float(pts[:, 0].mean()))
        if _lons:
            fig.plot(x=_lons, y=_lats, style="c0.20c", fill="orange",
                      pen="0.4p,black")

    fig.text(text=f"{int(time)} Ma  ({MODEL_NAME})  "
                   f"gaps/overlaps: {len(gaps)}  |  "
                   f"non-unique sub-segments: {len(nu_sections)}  |  "
                   f"missing polarity: {len(missing_pol)}",
              position="TL", offset="0.25c/-0.25c", justify="TL",
              font="10p,Helvetica-Bold,black", fill="white", pen="0.5p,gray40")

    if out_path is not None:
        fig.savefig(str(out_path), dpi=100)
        return None
    return fig

for _age in SNAPSHOT_AGES_MA:
    print(f"  rendering {_age} Ma ...")
    _f = render_anomaly_map(_age, width_cm=15)
    _f.show(width=900)


### How to read these maps

- **Red thick lines** — someone drew a plate polygon whose boundary uses this sub-segment, but the opposite side of that sub-segment either has no plate at all (gap) or has multiple plates trying to share it (overlap). This shows up visually as an isolated red line hanging off a plate boundary. Gaps are typically at plate triple junctions where the geometry doesn't quite close; overlaps are typically where two plate polygons cross each other by mistake.
- **Blue thick lines** — the same section reference appears more than once in a single plate polygon, and its two instances physically touch. Usually a construction bug where the modeller has added section A twice to the polygon boundary. (Head-of-A + B + tail-of-A + C without the head and tail touching is a legitimate pattern and DOESN'T get flagged.)
- **Orange dots** — subduction-zone features whose polarity attribute is missing or set to "Unknown". Feature-extraction workflows (see T62) can't determine which direction the slab dips at these features, so they get silently dropped from any subduction analysis.

**Watch for**

- **Clusters of red** at a particular time step → the modeller had trouble closing a specific triple junction at that age. Often a fix in one place clears many flags.
- **Persistent orange dots** at one location across multiple snapshots → a subduction zone that has never had its polarity set, easy fix.
- **Blue lines at 100 Ma but absent at 200 Ma** → the polygon boundary was constructed by a different modeller for different age windows, and only the younger window has the duplication bug.


## 4. Compact MP4 video at 1-Myr cadence over 100-200 Ma

Same rendering as §3 for every 1-Myr step. This is the most useful visualisation for spotting **transient** anomalies — flags that appear for only a handful of frames as topology changes through time are usually the hardest to notice from static snapshots but easiest to see in animation.


In [ ]:
FRAMES_DIR.mkdir(exist_ok=True, parents=True)
VIDEO_DIR.mkdir(exist_ok=True, parents=True)

_target_dpi = 100
_frame_width_cm = VIDEO_WIDTH_PX / _target_dpi * 2.54

ages_desc = np.arange(VIDEO_END_MA, VIDEO_START_MA + 1e-6, VIDEO_CADENCE_MA)[::-1]
print(f"  will render {len(ages_desc)} frames at ~{VIDEO_WIDTH_PX} px wide")

frame_paths = []
for i, _age in enumerate(ages_desc):
    _p = FRAMES_DIR / f"frame_{int(round(_age)):04d}Ma.png"
    if not _p.exists():
        render_anomaly_map(float(_age), out_path=_p, width_cm=_frame_width_cm)
    if i % 10 == 0:
        print(f"    frame {i+1}/{len(ages_desc)}  ({int(round(_age))} Ma)")
    frame_paths.append(_p)
print(f"  ✓ {len(frame_paths)} frames on disk")

import imageio.v2 as imageio
writer = imageio.get_writer(
    str(VIDEO_PATH), fps=VIDEO_FPS, codec="libx264",
    ffmpeg_params=["-crf", str(VIDEO_CRF), "-preset", "veryfast", "-pix_fmt", "yuv420p"],
)
for _p in frame_paths:
    writer.append_data(imageio.imread(str(_p)))
writer.close()
print(f"  ✓ wrote {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")

from IPython.display import Video
Video(str(VIDEO_PATH), embed=True, width=VIDEO_WIDTH_PX)


## Extend this

- **Deeper time windows.** The detectors work at any age the plate model covers. For Z22, that's 0-1800 Ma. Extending the window to the whole model reveals whether anomaly counts systematically increase with age (usually yes, because older topologies are less well-constrained).
- **Type-filter the results.** If your analysis only cares about MOR anomalies (not SZ or transform), filter the returned anomaly features by `.get_feature_type()` before rendering. Alternatively, filter by plate ID or geographic bounding box for regional debugging.
- **Export to GPML for GPlates.** Take the per-time anomaly features and write them as a `pygplates.FeatureCollection` to a `.gpml` file. Load it as an overlay in the GPlates desktop app for interactive inspection alongside the full plate model.
- **Compare against a fixed model.** After a modeller repairs a batch of anomalies, re-run this notebook against the new version and count the reduction. That's the standard QC metric.
- **Time-integrated anomaly hotspots.** Accumulate every anomaly location across the 1-Myr video into a single point cloud, then run DBSCAN — the resulting clusters are the "persistent anomaly regions" the modeller should prioritise for repair.

## Related resources

- Upstream repo: <https://github.com/EarthByte/plate-model-debug>. See the `AnomalyCaseStudy.pptx` under Topology-Construction-Anomalies for worked case studies of specific construction errors.
- GPlates desktop app: <https://www.gplates.org> — for interactive visualisation of anomaly GPMLs alongside the plate model.
- Companion notebooks: **T59** sign anomalies at boundaries, **T60** velocity magnitude at MORs, **T62** feature extractability at subduction zones.

## References

- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Fox, P. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9(2), 371-383. https://doi.org/10.1002/gdj3.146
- Mather, B.R., et al. (2024) Deep time spatio-temporal data analysis using pyGPlates with PlateTectonicTools and GPlately. *Applied Computing and Geosciences* 22, 100152.
